# Pipeline de datos: pedidos de una app de delivery con PySpark y MySQL

**Pregunta de negocio:** ¿en qué zonas y a qué horas se tarda más la entrega y se concentra la demanda de pedidos?

**Flujo del pipeline**

```
CSV (≥ 50,000 pedidos) → PySpark (limpieza, columnas calculadas, groupBy) → MySQL → SELECT de validación
```

| Fase | Contenido |
|---|---|
| 1 | Instalación, generación/carga del CSV y `SparkSession` |
| 2 | Esquema, limpieza, columnas calculadas, filtros y agregaciones |
| 3 | Conexión a MySQL, creación de la tabla e inserción con sentencias preparadas |
| 4 | Validación con `SELECT` y evidencia |

## Fase 1. Configuración del entorno y extracción

### 1.1 Instalación de dependencias

In [11]:
%pip install -q pyspark mysql-connector-python

### 1.2 (Opcional) Servidor MySQL local en Google Colab

Salta esta celda si ya tienes MySQL en tu equipo o en un servidor. En Colab no hay MySQL ni Java 17, así que se instalan aquí.
Cambia la contraseña antes de ejecutarla; se usará después como variable de entorno `MYSQL_PASSWORD`.

In [ ]:
%%bash
# Solo para Google Colab / Ubuntu. Ajusta la contraseña.
apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-17-jdk-headless mysql-server > /dev/null

# Detener cualquier instancia existente para evitar conflictos
service mysql stop

# Iniciar mysql de forma segura sin tablas de privilegios temporalmente
mysqld_safe --skip-grant-tables --skip-networking &
sleep 3

# Modificar el usuario root usando privilegios administrativos
mysql -u root -e "UPDATE mysql.user SET authentication_string=null WHERE User='root'; FLUSH PRIVILEGES;"
mysql -u root -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'CLAVE'; FLUSH PRIVILEGES;"

# Detener el servidor seguro y arrancar el servicio de forma normal
killall mysqld
sleep 2
service mysql start
sleep 2

 * Stopping MySQL database server mysqld
   ...done.
2026-10-16T21:32:06.702885Z mysqld_safe Logging to '/var/log/mysql/error.log'.
2026-10-16T21:32:06.729319Z mysqld_safe Starting mysqld daemon with databases from /var/lib/mysql
2026-10-16T21:32:10.430084Z mysqld_safe mysqld from pid file /var/lib/mysql/f25749baf092.pid ended
 * Starting MySQL database server mysqld
   ...done.


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
su: warning: cannot change directory to /nonexistent: No such file or directory


### 1.3 Importaciones y configuración general

Todos los parámetros ajustables (ruta del CSV, tamaño del dataset, credenciales) quedan juntos aquí.
La contraseña **no se escribe en el código**: se toma de la variable de entorno `MYSQL_PASSWORD` o se pide con `getpass`.

In [13]:
import csv
import getpass
import os
import random
from datetime import datetime, timedelta

import mysql.connector
from mysql.connector import Error as MySQLError
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType, IntegerType, StringType, StructField, StructType, TimestampType,
)

RUTA_CSV = "pedidos_delivery.csv"
N_PEDIDOS = 80_000          # el enunciado exige mínimo 50,000
SEMILLA = 2026
IVA = 0.13

MYSQL_CFG = {
    "host": os.getenv("MYSQL_HOST", "127.0.0.1"),
    "port": int(os.getenv("MYSQL_PORT", "3306")),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD") or getpass.getpass("Contraseña de MySQL: "),
}
BD = "delivery_analytics"
TABLA = "resumen_zona_hora"

Contraseña de MySQL: ··········


### 1.4 Generación del dataset (CSV de transacciones)

Como no se entrega un archivo, se genera un CSV sintético y reproducible (semilla fija) con 80,000 pedidos de 90 días.
A propósito se inyectan problemas de calidad para que la fase de limpieza tenga trabajo real:
montos vacíos, tiempos de entrega imposibles, zonas escritas distinto, calificaciones fuera de rango y pedidos duplicados.

Si prefieres un dataset real, basta con apuntar `RUTA_CSV` a un archivo con las mismas columnas.

In [14]:
ZONAS = ["Escalón", "San Benito", "Santa Tecla", "Soyapango", "Mejicanos", "Antiguo Cuscatlán"]
TIPOS = ["Pizza", "Pollo", "Hamburguesa", "Pupusas", "Sushi", "Postres"]
PAGOS = ["Tarjeta", "Efectivo", "Billetera"]
# Distribución de la demanda por hora (12-14 almuerzo, 18-21 cena)
HORAS = list(range(10, 24))
PESO_HORA = [2, 4, 9, 10, 8, 3, 3, 5, 10, 12, 11, 8, 4, 2]
BASE_ENTREGA = {"Escalón": 24, "San Benito": 22, "Santa Tecla": 30,
                "Soyapango": 38, "Mejicanos": 34, "Antiguo Cuscatlán": 27}


def generar_csv(ruta, n, semilla):
    rnd = random.Random(semilla)
    inicio = datetime(2026, 6, 1)
    filas = []
    for i in range(n):
        zona = rnd.choices(ZONAS, weights=[18, 14, 20, 22, 15, 11])[0]
        hora = rnd.choices(HORAS, weights=PESO_HORA)[0]
        momento = inicio + timedelta(days=rnd.randrange(90), hours=hora,
                                     minutes=rnd.randrange(60), seconds=rnd.randrange(60))
        extra_pico = 8 if hora in (13, 19, 20) else 0     # en hora pico se tarda más
        minutos = max(8, int(rnd.gauss(BASE_ENTREGA[zona] + extra_pico, 6)))
        filas.append({
            "id_pedido": f"PD{i + 1:06d}",
            "fecha_hora": momento.strftime("%Y-%m-%d %H:%M:%S"),
            "zona": zona,
            "tipo_comida": rnd.choice(TIPOS),
            "subtotal": round(rnd.uniform(4, 45), 2),
            "costo_envio": rnd.choice([0.0, 1.0, 1.5, 2.0, 2.5]),
            "metodo_pago": rnd.choices(PAGOS, weights=[50, 35, 15])[0],
            "minutos_entrega": minutos,
            "calificacion": rnd.choices([1, 2, 3, 4, 5], weights=[3, 5, 12, 35, 45])[0],
        })

    # --- Datos sucios intencionales ---
    def muestra(p):
        return rnd.sample(range(n), int(n * p))

    for i in muestra(0.02):  filas[i]["subtotal"] = ""
    for i in muestra(0.015): filas[i]["metodo_pago"] = ""
    for i in muestra(0.01):  filas[i]["minutos_entrega"] = rnd.choice([0, -5, 600])
    for i in muestra(0.01):  filas[i]["calificacion"] = rnd.choice([0, 9])
    for i in muestra(0.03):
        filas[i]["zona"] = rnd.choice([filas[i]["zona"].upper(), " " + filas[i]["zona"].lower() + " "])
    filas += [dict(filas[i]) for i in muestra(0.004)]      # duplicados exactos

    rnd.shuffle(filas)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
        w.writeheader()
        w.writerows(filas)


if not os.path.exists(RUTA_CSV):
    generar_csv(RUTA_CSV, N_PEDIDOS, SEMILLA)
print(f"CSV listo: {RUTA_CSV}  ({os.path.getsize(RUTA_CSV) / 1_048_576:.1f} MB)")

CSV listo: pedidos_delivery.csv  (5.5 MB)


### 1.5 Sesión de Spark

Se crea la `SparkSession` con un nombre descriptivo, `local[*]` (usa todos los núcleos como ejecutores) y
`spark.sql.shuffle.partitions` reducido a 8: el valor por defecto (200) es excesivo para este volumen.

In [15]:
spark = (
    SparkSession.builder
    .appName("Pipeline_Delivery_PySpark_a_MySQL")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "America/El_Salvador")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| paralelismo:", spark.sparkContext.defaultParallelism)

Spark 4.0.4 | paralelismo: 2


## Fase 2. Procesamiento distribuido con PySpark

### 2.1 Esquema explícito y lectura

Definir el esquema a mano evita la pasada extra de `inferSchema` y asegura los tipos correctos.
`mode="PERMISSIVE"` deja pasar filas problemáticas como nulos en lugar de romper la lectura.

In [16]:
esquema = StructType([
    StructField("id_pedido", StringType()),
    StructField("fecha_hora", TimestampType()),
    StructField("zona", StringType()),
    StructField("tipo_comida", StringType()),
    StructField("subtotal", DoubleType()),
    StructField("costo_envio", DoubleType()),
    StructField("metodo_pago", StringType()),
    StructField("minutos_entrega", IntegerType()),
    StructField("calificacion", IntegerType()),
])

crudo = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(esquema)
    .csv(RUTA_CSV)
    .repartition(8)
)
n_crudo = crudo.count()
print(f"Registros leídos: {n_crudo:,}  |  particiones: {crudo.rdd.getNumPartitions()}")
crudo.printSchema()
crudo.show(5, truncate=False)

Registros leídos: 80,320  |  particiones: 8
root
 |-- id_pedido: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- zona: string (nullable = true)
 |-- tipo_comida: string (nullable = true)
 |-- subtotal: double (nullable = true)
 |-- costo_envio: double (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- minutos_entrega: integer (nullable = true)
 |-- calificacion: integer (nullable = true)

+---------+-------------------+-----------------+-----------+--------+-----------+-----------+---------------+------------+
|id_pedido|fecha_hora         |zona             |tipo_comida|subtotal|costo_envio|metodo_pago|minutos_entrega|calificacion|
+---------+-------------------+-----------------+-----------+--------+-----------+-----------+---------------+------------+
|PD078081 |2026-07-26 20:20:55|Antiguo Cuscatlán|Sushi      |25.15   |1.0        |Billetera  |40             |4           |
|PD007936 |2026-07-30 19:19:11|Soyapango        |Sushi      |5.74    |

### 2.2 Diagnóstico de calidad: nulos por columna

In [17]:
crudo.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in crudo.columns]).show()

+---------+----------+----+-----------+--------+-----------+-----------+---------------+------------+
|id_pedido|fecha_hora|zona|tipo_comida|subtotal|costo_envio|metodo_pago|minutos_entrega|calificacion|
+---------+----------+----+-----------+--------+-----------+-----------+---------------+------------+
|        0|         0|   0|          0|    1605|          0|       1206|              0|           0|
+---------+----------+----+-----------+--------+-----------+-----------+---------------+------------+



### 2.3 Limpieza

| Problema | Tratamiento |
|---|---|
| Pedidos duplicados | `dropDuplicates` por `id_pedido` |
| Zona con mayúsculas/espacios distintos | `trim` + `initcap` |
| `subtotal` vacío | se descarta: sin monto no hay pedido analizable |
| `metodo_pago` vacío | se imputa como `"Sin dato"` (el pedido sí ocurrió) |
| Tiempos de entrega imposibles | se conservan solo los de 5 a 180 minutos |
| Calificación fuera de 1-5 | se anula (`null`) pero se conserva el pedido |

In [18]:
limpio = (
    crudo
    .dropDuplicates(["id_pedido"])
    .dropna(subset=["id_pedido", "fecha_hora", "zona", "subtotal"])
    .withColumn("zona", F.initcap(F.trim("zona")))
    .withColumn("metodo_pago", F.coalesce(F.nullif(F.trim("metodo_pago"), F.lit("")), F.lit("Sin dato")))
    .filter(F.col("minutos_entrega").between(5, 180))
    .withColumn("calificacion",
                F.when(F.col("calificacion").between(1, 5), F.col("calificacion")))
    .cache()
)
n_limpio = limpio.count()
print(f"Antes: {n_crudo:,} | Después: {n_limpio:,} | Descartados: {n_crudo - n_limpio:,}")
limpio.select("zona").distinct().orderBy("zona").show(truncate=False)

Antes: 80,320 | Después: 77,622 | Descartados: 2,698
+-----------------+
|zona             |
+-----------------+
|Antiguo Cuscatlán|
|Escalón          |
|Mejicanos        |
|San Benito       |
|Santa Tecla      |
|Soyapango        |
+-----------------+



### 2.4 Columnas calculadas

- `iva` y **`total_con_impuestos`** = (subtotal + IVA 13 %) + costo de envío.
- `hora` y `dia_semana` extraídos de la fecha (dimensiones de agrupación).
- `franja`: clasificación del momento del día con `when/otherwise`.
- `entrega_lenta`: bandera si el pedido tardó más de 45 minutos.

In [19]:
pedidos = (
    limpio
    .withColumn("iva", F.round(F.col("subtotal") * IVA, 2))
    .withColumn("total_con_impuestos",
                F.round(F.col("subtotal") + F.col("iva") + F.col("costo_envio"), 2))
    .withColumn("hora", F.hour("fecha_hora"))
    .withColumn("dia_semana", F.date_format("fecha_hora", "EEEE"))
    .withColumn("franja",
                F.when(F.col("hora").between(10, 11), "Mañana")
                 .when(F.col("hora").between(12, 15), "Almuerzo")
                 .when(F.col("hora").between(16, 17), "Tarde")
                 .when(F.col("hora").between(18, 21), "Cena")
                 .otherwise("Noche"))
    .withColumn("entrega_lenta", (F.col("minutos_entrega") > 45).cast("int"))
)
pedidos.select("id_pedido", "zona", "subtotal", "iva", "costo_envio",
               "total_con_impuestos", "hora", "franja", "entrega_lenta").show(6)

+---------+-----------------+--------+----+-----------+-------------------+----+--------+-------------+
|id_pedido|             zona|subtotal| iva|costo_envio|total_con_impuestos|hora|  franja|entrega_lenta|
+---------+-----------------+--------+----+-----------+-------------------+----+--------+-------------+
| PD000002|        Soyapango|   29.18|3.79|        0.0|              32.97|  20|    Cena|            0|
| PD000023|        Soyapango|   30.47|3.96|        2.0|              36.43|  13|Almuerzo|            1|
| PD000042|        Soyapango|   16.11|2.09|        1.5|               19.7|  19|    Cena|            0|
| PD000043|       San Benito|    37.3|4.85|        2.5|              44.65|  20|    Cena|            0|
| PD000053|      Santa Tecla|   27.74|3.61|        2.5|              33.85|  21|    Cena|            0|
| PD000056|Antiguo Cuscatlán|    7.46|0.97|        2.0|              10.43|  13|Almuerzo|            0|
+---------+-----------------+--------+----+-----------+---------

### 2.5 Filtros avanzados

Pedidos "problemáticos": entrega lenta **y** calificación baja (≤ 3) **o** sin calificación, en franja de cena,
pagados con tarjeta o billetera. Se resumen por zona para ver dónde duele más.

In [20]:
problematicos = pedidos.filter(
    (F.col("entrega_lenta") == 1)
    & (F.col("calificacion").isNull() | (F.col("calificacion") <= 3))
    & (F.col("franja") == "Cena")
    & F.col("metodo_pago").isin("Tarjeta", "Billetera")
)
n_prob = problematicos.count()
print(f"Pedidos problemáticos: {n_prob:,} ({n_prob / n_limpio:.2%} del total)")

(problematicos.groupBy("zona")
    .agg(F.count("*").alias("pedidos"),
         F.round(F.avg("minutos_entrega"), 1).alias("min_prom"))
    .orderBy(F.desc("pedidos"))
    .show())

Pedidos problemáticos: 523 (0.67% del total)
+-----------------+-------+--------+
|             zona|pedidos|min_prom|
+-----------------+-------+--------+
|        Soyapango|    322|    49.8|
|        Mejicanos|    123|    49.1|
|      Santa Tecla|     60|    48.1|
|Antiguo Cuscatlán|     12|    48.5|
|          Escalón|      5|    47.8|
|       San Benito|      1|    49.0|
+-----------------+-------+--------+



### 2.6 Agregaciones con `groupBy`

Primero una vista general por hora y luego la tabla que se guardará en MySQL:
**métricas por zona y hora**.

In [21]:
(pedidos.groupBy("hora")
    .agg(F.count("*").alias("pedidos"),
         F.round(F.avg("total_con_impuestos"), 2).alias("ticket_prom"),
         F.round(F.avg("minutos_entrega"), 1).alias("min_prom"))
    .orderBy("hora")
    .show(20))

+----+-------+-----------+--------+
|hora|pedidos|ticket_prom|min_prom|
+----+-------+-----------+--------+
|  10|   1736|      29.06|    29.3|
|  11|   3357|      29.19|    29.1|
|  12|   7745|      29.17|    29.2|
|  13|   8441|      29.42|    37.4|
|  14|   6889|      29.18|    29.4|
|  15|   2613|      29.09|    29.1|
|  16|   2516|      29.01|    29.1|
|  17|   4278|      29.19|    29.4|
|  18|   8316|      28.64|    29.3|
|  19|  10180|      29.14|    37.4|
|  20|   9541|      29.27|    37.4|
|  21|   6881|      28.91|    29.4|
|  22|   3486|      29.41|    29.2|
|  23|   1643|      28.97|    29.3|
+----+-------+-----------+--------+



In [22]:
resumen = (
    pedidos.groupBy("zona", "hora")
    .agg(
        F.count("*").alias("total_pedidos"),
        F.round(F.avg("total_con_impuestos"), 2).alias("ticket_promedio"),
        F.round(F.sum("total_con_impuestos"), 2).alias("ingresos"),
        F.round(F.avg("minutos_entrega"), 1).alias("minutos_promedio"),
        F.round(F.avg("entrega_lenta") * 100, 1).alias("pct_entregas_lentas"),
    )
    .orderBy("zona", "hora")
    .cache()
)
print("Filas a persistir:", resumen.count())
resumen.show(10)

Filas a persistir: 84
+-----------------+----+-------------+---------------+--------+----------------+-------------------+
|             zona|hora|total_pedidos|ticket_promedio|ingresos|minutos_promedio|pct_entregas_lentas|
+-----------------+----+-------------+---------------+--------+----------------+-------------------+
|Antiguo Cuscatlán|  10|          187|          28.42| 5314.77|            26.1|                0.0|
|Antiguo Cuscatlán|  11|          376|          29.47|11079.58|            26.9|                0.0|
|Antiguo Cuscatlán|  12|          890|           29.4|26167.48|            26.1|                0.1|
|Antiguo Cuscatlán|  13|          895|          28.88|25848.49|            34.7|                3.7|
|Antiguo Cuscatlán|  14|          761|          28.97|22044.09|            26.4|                0.1|
|Antiguo Cuscatlán|  15|          303|          29.59| 8965.79|            26.0|                0.0|
|Antiguo Cuscatlán|  16|          268|          29.48| 7899.44|      

## Fase 3. Persistencia en MySQL

### 3.1 Conexión
Se abre la conexión con `mysql-connector-python`, se crea la base si no existe y se selecciona.
Se usa `utf8mb4` para guardar bien los acentos (Escalón, Antiguo Cuscatlán).

In [23]:
conn = None
try:
    conn = mysql.connector.connect(**MYSQL_CFG, charset="utf8mb4", use_pure=True)
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE IF NOT EXISTS {BD} CHARACTER SET utf8mb4")
    conn.database = BD
    cur.execute("SELECT VERSION()")
    print("Conectado a MySQL", cur.fetchone()[0], "| BD:", BD)
except MySQLError as e:
    print("Error de conexión:", e)
    raise

Conectado a MySQL 8.0.46-0ubuntu0.24.04.4 | BD: delivery_analytics


### 3.2 Modelado de la tabla

Una fila por **zona + hora**. La llave única `(zona, hora)` hace que re-ejecutar el notebook
actualice los datos en vez de duplicarlos (carga idempotente).

In [24]:
cur.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLA} (
    id_registro         INT AUTO_INCREMENT PRIMARY KEY,
    zona                VARCHAR(40)   NOT NULL,
    hora                TINYINT       NOT NULL,
    total_pedidos       INT           NOT NULL,
    ticket_promedio     DECIMAL(8,2)  NOT NULL,
    ingresos            DECIMAL(12,2) NOT NULL,
    minutos_promedio    DECIMAL(6,1)  NOT NULL,
    pct_entregas_lentas DECIMAL(5,1)  NOT NULL,
    cargado_en          TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    UNIQUE KEY uq_zona_hora (zona, hora)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
""")
conn.commit()
print(f"Tabla {TABLA} lista.")

Tabla resumen_zona_hora lista.


### 3.3 Carga con sentencias preparadas

El resultado de Spark ya está agregado (pocas filas), así que se trae al driver con `collect()` y se convierte
a tuplas de tipos nativos de Python. Se usa un cursor **preparado** (`prepared=True`) con marcadores de posición:
los valores viajan separados de la consulta, lo que evita inyección SQL. Si algo falla, se hace `rollback`.

In [25]:
filas = [
    (r["zona"], int(r["hora"]), int(r["total_pedidos"]), float(r["ticket_promedio"]),
     float(r["ingresos"]), float(r["minutos_promedio"]), float(r["pct_entregas_lentas"]))
    for r in resumen.collect()
]

SQL_UPSERT = f"""
INSERT INTO {TABLA}
    (zona, hora, total_pedidos, ticket_promedio, ingresos, minutos_promedio, pct_entregas_lentas)
VALUES (%s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    total_pedidos = VALUES(total_pedidos),
    ticket_promedio = VALUES(ticket_promedio),
    ingresos = VALUES(ingresos),
    minutos_promedio = VALUES(minutos_promedio),
    pct_entregas_lentas = VALUES(pct_entregas_lentas)
"""

cur_prep = conn.cursor(prepared=True)
try:
    for fila in filas:
        cur_prep.execute(SQL_UPSERT, fila)
    conn.commit()
    print(f"Filas enviadas a MySQL: {len(filas)}")
except MySQLError as e:
    conn.rollback()
    print("Falló la carga, rollback aplicado:", e)
    raise
finally:
    cur_prep.close()

Filas enviadas a MySQL: 84


## Fase 4. Validación

Tres comprobaciones: (1) el conteo de filas en MySQL coincide con el de Spark,
(2) una muestra con `SELECT`, (3) un resumen calculado directamente en SQL.

In [26]:
cur.execute(f"SELECT COUNT(*) FROM {TABLA}")
en_mysql = cur.fetchone()[0]
en_spark = resumen.count()
print(f"Spark: {en_spark} filas | MySQL: {en_mysql} filas ->",
      "COINCIDEN" if en_mysql == en_spark else "NO COINCIDEN")

cur.execute(f"""
    SELECT zona, hora, total_pedidos, ticket_promedio, minutos_promedio, pct_entregas_lentas
    FROM {TABLA}
    ORDER BY total_pedidos DESC
    LIMIT 10
""")
print("\nTop 10 zona-hora por volumen de pedidos")
print(f"{'zona':<20}{'hora':>5}{'pedidos':>9}{'ticket':>9}{'min':>7}{'%lentas':>9}")
for z, h, p, t, m, l in cur.fetchall():
    print(f"{z:<20}{h:>5}{p:>9}{float(t):>9.2f}{float(m):>7.1f}{float(l):>9.1f}")

Spark: 84 filas | MySQL: 84 filas -> COINCIDEN

Top 10 zona-hora por volumen de pedidos
zona                 hora  pedidos   ticket    min  %lentas
Soyapango              19     2258    29.46   45.4     49.1
Soyapango              20     2097    28.98   45.6     51.0
Santa Tecla            19     2013    28.97   37.5     10.0
Soyapango              13     1915    29.32   45.5     49.5
Santa Tecla            20     1835    29.21   37.5      9.8
Soyapango              18     1818    28.51   37.5      8.1
Escalón                19     1788    29.09   31.5      0.7
Escalón                20     1780    29.26   31.5      0.8
Soyapango              12     1719    29.12   37.4      9.5
Santa Tecla            18     1673    28.65   29.6      0.4


In [27]:
cur.execute(f"""
    SELECT zona,
           SUM(total_pedidos)                                   AS pedidos,
           ROUND(SUM(ingresos), 2)                              AS ingresos,
           ROUND(SUM(minutos_promedio * total_pedidos) / SUM(total_pedidos), 1) AS min_ponderado
    FROM {TABLA}
    GROUP BY zona
    ORDER BY min_ponderado DESC
""")
print(f"{'zona':<20}{'pedidos':>9}{'ingresos':>13}{'min_pond':>10}")
for z, p, i, m in cur.fetchall():
    print(f"{z:<20}{int(p):>9}{float(i):>13,.2f}{float(m):>10.1f}")

zona                  pedidos     ingresos  min_pond
Soyapango               17179   499,742.99      40.4
Mejicanos               11724   342,062.39      36.5
Santa Tecla             15419   447,899.99      32.4
Antiguo Cuscatlán        8551   249,165.07      29.3
Escalón                 13876   405,591.72      26.4
San Benito              10873   316,243.70      24.4


### Evidencia para la entrega

Ejecuta esta consulta en **MySQL Workbench** o en la terminal y toma la captura de pantalla de la tabla con los datos insertados:

```sql
USE delivery_analytics;
SELECT COUNT(*) AS filas FROM resumen_zona_hora;
SELECT zona, hora, total_pedidos, ticket_promedio, ingresos
FROM resumen_zona_hora ORDER BY zona, hora LIMIT 15;
```

In [ ]:
!mysql -uroot -p'CLAVE' -e "SELECT COUNT(*) AS filas FROM delivery_analytics.resumen_zona_hora; SELECT zona, hora, total_pedidos, ticket_promedio, ingresos FROM delivery_analytics.resumen_zona_hora ORDER BY zona, hora LIMIT 15;" 2>/dev/null

+-------+
| filas |
+-------+
|    84 |
+-------+
+--------------------+------+---------------+-----------------+----------+
| zona               | hora | total_pedidos | ticket_promedio | ingresos |
+--------------------+------+---------------+-----------------+----------+
| Antiguo Cuscatlán  |   10 |           187 |           28.42 |  5314.77 |
| Antiguo Cuscatlán  |   11 |           376 |           29.47 | 11079.58 |
| Antiguo Cuscatlán  |   12 |           890 |           29.40 | 26167.48 |
| Antiguo Cuscatlán  |   13 |           895 |           28.88 | 25848.49 |
| Antiguo Cuscatlán  |   14 |           761 |           28.97 | 22044.09 |
| Antiguo Cuscatlán  |   15 |           303 |           29.59 |  8965.79 |
| Antiguo Cuscatlán  |   16 |           268 |           29.48 |  7899.44 |
| Antiguo Cuscatlán  |   17 |           511 |           29.20 | 14922.78 |
| Antiguo Cuscatlán  |   18 |           924 |           28.95 | 26746.01 |
| Antiguo Cuscatlán  |   19 |          1089 |     

### Cierre de recursos

In [28]:
try:
    cur.close()
    if conn is not None and conn.is_connected():
        conn.close()
    print("Conexión MySQL cerrada.")
finally:
    spark.stop()
    print("SparkSession detenida.")

Conexión MySQL cerrada.
SparkSession detenida.


## Conclusiones

A partir del procesamiento distribuido con PySpark y el almacenamiento analítico en MySQL, se obtienen los siguientes hallazgos cuantitativos del negocio:

1. **Calidad de datos y depuración:**
   - De los **80,320** registros brutos leídos, se conservaron **77,622** pedidos válidos y se descartaron **2,698** registros (un **3.36 %** del total).
   - Los motivos principales de descarte fueron duplicados por `id_pedido`, valores nulos en campos obligatorios (1,605 subtotales vacíos) y tiempos de entrega anómalos fuera del rango operativo de 5 a 180 minutos (valores erróneos como 0, -5 y 600 min). Además, se estandarizaron los nombres de las zonas (`trim` + `initcap`) y se imputó el método de pago faltante como `"Sin dato"`.

2. **Horas con mayor demanda:**
   - La actividad se concentra en las franjas de **Cena** y **Almuerzo**:
     - **1.ª Hora pico:** Las **19:00 (7:00 PM)** con **10,180 pedidos** (Franja Cena).
     - **2.ª Hora pico:** Las **20:00 (8:00 PM)** con **9,541 pedidos** (Franja Cena).
     - **3.ª Hora pico:** Las **13:00 (1:00 PM)** con **8,441 pedidos** (Franja Almuerzo).
     - Seguidas por las **18:00** (8,316 pedidos) y las **12:00** (7,745 pedidos).

3. **Zona con mayor tiempo de entrega:**
   - **Soyapango** presentó el mayor tiempo promedio ponderado de entrega con **40.4 minutos**, seguida de **Mejicanos** (**36.5 min**) y **Santa Tecla** (**32.4 min**). Las zonas con entregas más rápidas fueron **Escalón** (**26.4 min**) y **San Benito** (**24.4 min**).
   - **Efecto de la hora pico:** En Soyapango, durante las 19:00 y 20:00 el tiempo promedio subió hasta **45.4 – 45.6 minutos**, registrando hasta un **51.0 % de entregas lentas** (> 45 min). Esto la convierte en la zona de mayor atención operativa, concentrando **322 de los 523 pedidos problemáticos** (61.6 % del total).

4. **Zona con más ingresos:**
   - **Soyapango** fue la zona con mayor facturación global con **$499,742.99** generados a partir de **17,179 pedidos**, coincidiendo plenamente con la zona de mayor volumen de transacciones.
   - El ranking total de facturación por zona fue:
     1. **Soyapango:** $499,742.99 (17,179 pedidos)
     2. **Santa Tecla:** $447,899.99 (15,419 pedidos)
     3. **Escalón:** $405,591.72 (13,876 pedidos)
     4. **Mejicanos:** $342,062.39 (11,724 pedidos)
     5. **San Benito:** $316,243.70 (10,873 pedidos)
     6. **Antiguo Cuscatlán:** $249,165.07 (8,551 pedidos)
   - El ticket promedio se mantuvo uniforme entre **$28.50 y $29.50** en todas las zonas, evidenciando que el liderazgo en ingresos responde a la alta densidad de demanda y no a un mayor precio por pedido.

5. **Persistencia e integridad:**
   - Se confirmó una correspondencia exacta de **84 filas** (6 zonas × 14 horas operativas de 10:00 a 23:00) entre la agregación distribuida en PySpark y la tabla `resumen_zona_hora` persistida en MySQL.

*Nota:* Los datos procesados corresponden a un dataset sintético para validar el pipeline de ingeniería y análisis de datos de principio a fin.
